In [0]:
%run ./00_common_dq_functions

In [0]:
from pyspark.sql.functions import col, to_timestamp

In [0]:
spark.sql("USE CATALOG pharma_medallion_v2")
spark.sql("USE SCHEMA dev_bronze")

DataFrame[]

In [0]:
bronze_db = "dev_bronze"
silver_db = "dev_silver"


DataFrame[]

In [0]:
bh_bz = spark.table(f"{bronze_db}.batch_header_bz")

bh_std = (bh_bz
    .withColumn("batch_id", col("batch_id").cast("string"))
)

bh_good1, bh_bad1 = dq_not_null(bh_std, ["batch_id"])
bh_sl,    bh_bad2 = dq_dedup(bh_good1, ["batch_id"])

bh_bad_all = dq_union_bad(bh_bad1, bh_bad2)
dq_write_quarantine(bh_bad_all, f"{silver_db}.dq_quarantine_batch_header")

bh_sl.write.mode("overwrite").format("delta").saveAsTable(f"{silver_db}.batch_header_sl")

In [0]:
ev_bz = spark.table(f"{bronze_db}.fact_events_src_bz")

# Standardize types
ev_std = (ev_bz
    .withColumn("batch_id", col("batch_id").cast("string"))
    .withColumn("event_type", col("event_type").cast("string"))
    .withColumn("event_time", to_timestamp(col("event_time")))
)


for ncol in ["event_value", "qty", "amount"]:
    if ncol in ev_std.columns:
        ev_std = ev_std.withColumn(ncol, col(ncol).cast("double"))

# 2.1 Not null checks
ev_good1, ev_bad1 = dq_not_null(ev_std, ["batch_id", "event_type", "event_time"])

# 2.2 Dedup checks
ev_good2, ev_bad2 = dq_dedup(ev_good1, ["batch_id", "event_time", "event_type"])

# 2.3 Timeliness check (no future timestamps)
ev_good3, ev_bad3 = dq_timeliness_no_future(ev_good2, "event_time")

# 2.4 FK check: batch_id exists in batch_header_sl
bh_sl_df = spark.table(f"{silver_db}.batch_header_sl")
ev_sl, ev_bad4 = dq_fk_exists(ev_good3, "batch_id", bh_sl_df, "batch_id")

# Union all event bad rows + write quarantine
ev_bad_all = dq_union_bad(ev_bad1, ev_bad2, ev_bad3, ev_bad4)
dq_write_quarantine(ev_bad_all, f"{silver_db}.dq_quarantine_fact_events_src")

# Write Silver fact table
ev_sl.write.mode("overwrite").format("delta").saveAsTable(f"{silver_db}.fact_events_src_sl")


In [0]:
print("batch_header_sl:", spark.table(f"{silver_db}.batch_header_sl").count())
print("dq_quarantine_batch_header:", spark.table(f"{silver_db}.dq_quarantine_batch_header").count())

print("fact_events_src_sl:", spark.table(f"{silver_db}.fact_events_src_sl").count())
print("dq_quarantine_fact_events_src:", spark.table(f"{silver_db}.dq_quarantine_fact_events_src").count())

batch_header_sl: 39
dq_quarantine_batch_header: 1
fact_events_src_sl: 136
dq_quarantine_fact_events_src: 3
